# Tutorial 09 — Linear pairwise smoothers: six algorithms, one estimate

On a **linear-Gaussian pairwise** state-space model, fixed-interval smoothing
$p(X_n \mid y_{1:N})$ has a single exact (MMSE) answer. The library ships **six**
classical ways to compute it — RTS, BF, MBF, MF, DWY and a variational/lifted
form — all behind one façade `Linear_PKS(param, method=...)`.

This tutorial presents each briefly and shows, on several linear models, that
they return **the same** smoothed mean and covariance down to round-off
($\approx 10^{-15}$). It is the notebook companion of the non-regression check
`python -m prg.run_dwy_equivalence` and of report §2.

In [ ]:
import sys
from pathlib import Path

# Make `prg` importable from the notebooks directory
REPO_ROOT = Path.cwd().parent
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
%matplotlib inline

from prg.classes.param_linear import ParamLinear
from prg.models.linear import ModelFactoryLinear
from prg.classes.linear_pks import Linear_PKS   # facade: method = RTS|BF|MBF|MF|DWY|VAR


def make_param_linear(model_name):
    m = ModelFactoryLinear.create(model_name)
    p = m.get_params().copy()
    dim_x = p.pop('dim_x'); dim_y = p.pop('dim_y')
    return ParamLinear(0, dim_x, dim_y, **p)


METHODS = ["RTS", "BF", "MBF", "MF", "DWY", "VAR"]
SEED = 42

## 1. The six algorithms

Model: $Z_{n+1} = A_{n+1} Z_n + B_{n+1} W_{n+1}$, couple $Z=(X,Y)$ markovian, with
$Y$ observed and $X$ latent. **All variants compute the exact posterior**
$p(X_n \mid y_{1:N})$; they differ only in *how* the computation is organized.

| `method` | Name | Idea |
|----------|------|------|
| `RTS` | Rauch–Tung–Striebel | forward filter, then a **chained** backward recursion (smoothing gain $G_n$) |
| `BF`  | Bryson–Frazier | backward **adjoint** $(\mu_n, N_n)$ at the couple level; the estimate is *recovered* from them |
| `MBF` | Modified BF (Bierman) | filtered adjoint $(\lambda_n, \Lambda_n)$ in dim $p$, **manifestly PSD** (no fragile subtraction) |
| `MF`  | Mayne–Fraser (two-filter) | forward filter **×** backward filter, fused in information form — the two passes are **independent** (parallelizable) |
| `DWY` | Desai–Weinert–Yusypchuk | backward filter on the **time-reversed** chain, then a forward recursion — the **dual** of RTS |
| `VAR` | Variational / lifted | the whole smoothed trajectory as **one block-tridiagonal linear solve** (the QP the five recursions solve implicitly) |

Each is also a dedicated class: `Linear_PKS_RTS`, `Linear_PKS_BF`, `Linear_PKS_MBF`,
`Linear_PKS_MF`, `Linear_PKS_DWY`, `Linear_PKS_VAR`. The derivations are in report §2.

## 2. They all reach the same estimate

Taking **RTS** as the reference, we measure the worst step-wise deviation of each
variant's smoothed **mean** and **covariance** over a full trajectory, on three
linear pairwise models. Every entry should sit at the double-precision round-off
level.

In [ ]:
def smoothed(model_name, method, N=120, seed=SEED):
    """Smoothed means (N+1, p) and covariances (N+1, p, p) for a given method."""
    pks = Linear_PKS(make_param_linear(model_name), sKey=seed, method=method)
    pks.process_N_data_smoother(N=N)
    X = np.array([np.asarray(h["Xkp1_smooth"], float).ravel() for h in pks.history])
    P = np.array([np.asarray(h["PXXkp1_smooth"], float) for h in pks.history])
    return X, P


MODELS = ["model_x1_y1_AQ_pairwise", "model_x2_y2_AQ_pairwise", "model_x3_y1_AQ_pairwise"]

rows = []
for mdl in MODELS:
    Xr, Pr = smoothed(mdl, "RTS")                      # RTS = reference
    for me in METHODS[1:]:
        Xv, Pv = smoothed(mdl, me)
        rows.append({
            "model": mdl.replace("model_", "").replace("_AQ_pairwise", ""),
            "method": me,
            "max|d mean| vs RTS": np.abs(Xv - Xr).max(),
            "max|d cov| vs RTS":  np.abs(Pv - Pr).max(),
        })

pd.set_option("display.float_format", lambda v: f"{v:.1e}")
pd.DataFrame(rows)

Every deviation is $\sim 10^{-15}$ on the means and $\sim 10^{-16}$ on the
covariances — i.e. **pure round-off**. The six algorithms are *numerically
indistinguishable*.

Overlaid on a single trajectory, their six smoothed means collapse onto one
curve, visibly closer to the truth than the filter:

In [ ]:
mdl = "model_x1_y1_AQ_pairwise"
ref = Linear_PKS(make_param_linear(mdl), sKey=SEED, method="RTS")
res = ref.process_N_data_smoother(N=120)
k  = np.array([r[0] for r in res])
xt = np.array([r[1].ravel() for r in res])[:, 0]   # true X
xf = np.array([r[4].ravel() for r in res])[:, 0]   # filtered X

fig, ax = plt.subplots(figsize=(11, 3.8))
ax.plot(k, xt, "k-", lw=1.5, label="true X")
ax.plot(k, xf, color="tab:orange", lw=1.0, alpha=0.6, label="filter (PKF)")
for me, c in zip(METHODS, plt.cm.viridis(np.linspace(0.0, 0.9, len(METHODS)))):
    Xv, _ = smoothed(mdl, me)
    ax.plot(k, Xv[:, 0], color=c, lw=1.3, ls="--", label=f"smoother {me}")
ax.set(title="Six linear smoothers — overlaid estimates (indistinguishable)",
       xlabel="n", ylabel="X")
ax.legend(ncol=4, fontsize=8, loc="best")
fig.tight_layout(); plt.show()

## 3. Same estimate — so which one?

Since the six are numerically identical, the choice is purely about **numerical
structure and cost**, not accuracy:

- **RTS / MBF** — cheapest (two passes, no prior moments). **MBF** is manifestly
  PSD, preferred when conditioning matters.
- **MF** — the only variant that *amortizes* its extra work: its two filters are
  **independent**, hence **parallelizable** (RTS is strictly sequential); its
  information form also handles diffuse priors.
- **DWY** — exact time-reversed **dual** of RTS; mostly of structural interest
  (in practice dominated by RTS).
- **VAR** — the *direct* batch view: a single sparse solve that unifies the
  recursions.

> **MF** and **DWY** additionally need the prior moments $(m_n, \Sigma_n)$
> (Lyapunov recursion) — a forward pre-pass without analogue in the forward
> filter, but data-independent and amortizable. See report §2.6.

## Going further

- **Report §2** — full derivations of the six linear smoothers and the
  equivalence proofs (RTS $\equiv$ BF $\equiv$ MBF $\equiv$ MF $\equiv$ DWY).
- **Non-regression**: `python -m prg.run_dwy_equivalence`, and
  `pytest prg/tests/test_linear_pks.py -k equals_rts`.
- **Nonlinear & particle smoothers** (EPKS, UPKS, UKS, PPS):
  `tutorial_07_smoothers.ipynb`.